In [21]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict,Annotated
from dotenv import load_dotenv
import os
load_dotenv()
from langchain_groq import ChatGroq

In [22]:
model = ChatGroq(model="openai/gpt-oss-120b", api_key=os.getenv("GROQ_API_KEY"))

In [23]:
class BatsmanState(TypedDict):
    balls_faced: int
    runs_scored: int
    fours_hit: int
    sixes_hit: int
    strike_rate: Annotated[float, lambda x, y: y]  # Take last value
    balls_per_boundary: Annotated[float, lambda x, y: y]
    boundary_percentage: Annotated[float, lambda x, y: y]
    summary: str


In [27]:
def calculate_strike_rate(state: BatsmanState) -> dict:
    return {'strike_rate': (state['runs_scored'] / state['balls_faced']) * 100}

def calculate_balls_per_boundary(state: BatsmanState) -> dict:
    total_boundaries = state['fours_hit'] + state['sixes_hit']
    bpb = state['balls_faced'] / total_boundaries if total_boundaries > 0 else float('inf')
    return {'balls_per_boundary': bpb}

def calculate_boundary_percentage(state: BatsmanState) -> dict:
    if state['balls_faced'] > 0:
        bp = ((state['fours_hit'] + state['sixes_hit']) / state['balls_faced']) * 100
    else:
        bp = 0.0
    return {'boundary_percentage': bp}

def define_summary(state: BatsmanState) -> dict:
    prompt = f"""
    The batsman faced {state['balls_faced']} balls and scored {state['runs_scored']} runs, hitting {state['fours_hit']} fours and {state['sixes_hit']} sixes. 
    The strike rate is {state['strike_rate']:.2f}, with a boundary percentage of {state['boundary_percentage']:.2f}% and {state['balls_per_boundary']:.2f} balls per boundary.
    Provide a concise summary of the batsman's performance in a t20 format.
    """
    response = model.invoke(prompt)
    return {'summary': response.content.strip()}

In [28]:
graph = StateGraph(BatsmanState)


graph.add_node("calculate_strike_rate", calculate_strike_rate)
graph.add_node("calculate_balls_per_boundary", calculate_balls_per_boundary)
graph.add_node("calculate_boundary_percentage", calculate_boundary_percentage)
graph.add_node("define_summary", define_summary)

graph.add_edge(START, "calculate_strike_rate")
graph.add_edge(START, "calculate_balls_per_boundary")
graph.add_edge(START, "calculate_boundary_percentage")
graph.add_edge("calculate_strike_rate", "define_summary")
graph.add_edge("calculate_balls_per_boundary", "define_summary")
graph.add_edge("calculate_boundary_percentage", "define_summary")
graph.add_edge("define_summary", END)

workflow = graph.compile()



In [29]:
initial_state = BatsmanState(
    balls_faced=120,
    runs_scored=80,
    fours_hit=10,
    sixes_hit=5
)

final_state = workflow.invoke(initial_state)
print("Final State:")
print(final_state)

Final State:
{'balls_faced': 120, 'runs_scored': 80, 'fours_hit': 10, 'sixes_hit': 5, 'strike_rate': 66.66666666666666, 'balls_per_boundary': 8.0, 'boundary_percentage': 12.5, 'summary': '**T20 Performance Summary**\n\n- **Runs:** 80 off 120 balls  \n- **Strike Rate:** 66.67 (well below typical T20 tempo)  \n- **Boundaries:** 10 fours & 5 sixes (15 boundaries)  \n- **Boundary Impact:**  \n  - **Boundary %:** 12.5\u202f% of the total runs came from boundaries  \n  - **Balls per Boundary:** 8.0 balls per boundary  \n\n**Interpretation:** The innings was very slow for a T20, with a low strike rate and a modest boundary contribution, indicating a defensive or struggling knock rather than a match‑winning effort.'}
